# Ingresos y áreas verdes por habitante en la RM
Ejecuta las celdas en orden. Al final obtienes el gráfico interactivo y el archivo HTML para descargar.

In [ ]:
# 1) Librerías y carga de archivos
import glob, json, unicodedata
import pandas as pd

try:
    from google.colab import files
    print("Sube los dos archivos: el Excel de áreas verdes y el CSV de la base de datos")
    files.upload()
except ImportError:
    print("Fuera de Colab: se usarán los archivos de la carpeta actual")

ruta_xlsx = glob.glob("*.xlsx")[0]
ruta_csv  = glob.glob("*.csv")[0]
print("Excel:", ruta_xlsx, "| CSV:", ruta_csv)

In [ ]:
# 2) Unir los datos por comuna
norm = lambda t: unicodedata.normalize("NFC", str(t)).strip().upper()

verdes = pd.read_excel(ruta_xlsx)
verdes["k"] = verdes["COMUNA"].map(norm)
verdes["m2"] = verdes["VALOR DEL INDICADOR EN M2"].astype(str).str.replace(",", ".").astype(float)

base = pd.read_csv(ruta_csv, sep=";")
base["k"] = base["Comuna"].map(norm)
ingreso = base[base["Indicador"] == "Ingreso_imponible_AFC_2020"].set_index("k")

CATEGORIAS = ["Alta Prioridad Social", "Media Alta Prioridad Social",
              "Media Baja Prioridad Social", "Baja Prioridad Social", "Sin Prioridad Social"]

filas = []
for _, f in verdes.iterrows():
    if f["k"] in ingreso.index:
        b = ingreso.loc[f["k"]]
        # Ingreso promedio = ingreso imponible AFC 2020 / 10 (igual que en el código original)
        filas.append([b["Comuna"], round(float(b["Valor"]) / 10), float(f["m2"]),
                      CATEGORIAS.index(b["Categoria_IPS_2022"])])

print(len(filas), "comunas unidas")
sin_datos = sorted(set(verdes["k"]) - set(ingreso.index))
if sin_datos: print("Sin coincidencia:", sin_datos)
pd.DataFrame(filas, columns=["Comuna", "Ingreso_Promedio", "M2_Area_Verde_Hab", "Categoria"]).head()

In [ ]:
# 3) Plantilla del gráfico (HTML + D3), no necesitas modificarla
PLANTILLA = r"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>Ingresos y áreas verdes por habitante en la RM</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Figtree:wght@400;600;800&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/d3/7.8.5/d3.min.js"></script>
<style>
:root{--rojo:#602C2C;--amarillo:#F6E7A6;--olive:#4F6815;--crema:#FAF7F2;color-scheme:light;
  box-sizing:border-box;padding-top:env(safe-area-inset-top,0px);padding-bottom:env(safe-area-inset-bottom,0px)}
html{scroll-padding-top:env(safe-area-inset-top,0px)}
body{margin:0;background:var(--crema);color:var(--rojo);font-family:Figtree,-apple-system,"Segoe UI",Roboto,Helvetica,Arial,sans-serif}
main{max-width:980px;margin:0 auto;padding:28px 20px 36px}
.head{display:flex;gap:16px;align-items:center}
.leaf{flex:none;width:46px;height:46px;background:var(--amarillo);border-radius:50% 50% 0 50%;display:grid;place-items:center;font-weight:800;font-size:22px}
h1{margin:0;font-size:clamp(20px,3.4vw,28px);line-height:1.15;color:var(--olive);font-weight:800}
.sub{margin:4px 0 0;font-size:14px;color:var(--rojo);opacity:.85}
.legend{display:flex;flex-wrap:wrap;gap:8px;margin:22px 0 6px}
.chip{display:inline-flex;align-items:center;gap:8px;padding:6px 12px 6px 8px;border:1.5px solid var(--rojo);border-radius:999px;background:transparent;color:var(--rojo);font:600 12px Figtree,sans-serif;cursor:pointer;transition:opacity .2s,background .2s}
.chip:hover{background:rgba(96,44,44,.07)}
.chip.off{opacity:.38}
.chip svg{display:block}
#chart{position:relative;width:100%}
svg.main{display:block;width:100%;height:auto;overflow:visible}
.coin{cursor:pointer;transition:opacity .2s}
.axis text{fill:var(--rojo);font:600 11px Figtree,sans-serif}
.axis .domain{stroke:var(--rojo)}
.axis .tick line{stroke:var(--rojo)}
.grid line{stroke:rgba(96,44,44,.12)}
.grid .domain{display:none}
.atitle{fill:var(--rojo);font:600 12px Figtree,sans-serif}
.who{fill:var(--olive);font:800 11px Figtree,sans-serif}
#tip{position:absolute;pointer-events:none;opacity:0;transition:opacity .12s;background:var(--rojo);color:var(--crema);border:1px solid var(--olive);border-radius:8px;padding:9px 12px;font-size:12px;line-height:1.5;box-shadow:0 4px 12px rgba(0,0,0,.18);min-width:170px;z-index:5}
#tip b{display:block;font-size:14px;margin-bottom:3px}
#tip span{color:var(--amarillo);font-weight:600}
.note{margin:14px 0 0;font-size:12px;opacity:.75;line-height:1.5}
</style>
</head>
<body>
<main>
  <div class="head">
    <div class="leaf">$</div>
    <div>
      <h1>Relación entre ingresos y áreas verdes por habitante en la RM</h1>
      <p class="sub">41 comunas urbanas · cada moneda es una comuna, su color indica la prioridad social (IPS 2022)</p>
    </div>
  </div>
  <div class="legend" id="legend" role="group" aria-label="Filtrar por categoría IPS 2022"></div>
  <div id="chart"><div id="tip"></div></div>
  <p class="note">Haz clic en una categoría para mostrarla u ocultarla y pasa el cursor sobre una moneda para ver el detalle. Las monedas se desplazan unos pocos píxeles para que no se superpongan; el valor exacto aparece siempre en el detalle. Ingreso promedio = ingreso imponible AFC 2020 ÷ 10.</p>
</main>
<script>
// [Comuna, Ingreso promedio (CLP), m² de área verde por habitante, Categoría IPS 2022]
const RAW=__DATOS__;
const CATS=[
 {n:"Alta Prioridad Social",c:"#602C2C",ink:"#FAF7F2"},
 {n:"Media Alta Prioridad Social",c:"#8C4334",ink:"#FAF7F2"},
 {n:"Media Baja Prioridad Social",c:"#C29243",ink:"#602C2C"},
 {n:"Baja Prioridad Social",c:"#F6E7A6",ink:"#602C2C"},
 {n:"Sin Prioridad Social",c:"#4F6815",ink:"#F6E7A6"}];
const DATA=RAW.map(([comuna,ing,m2,k])=>({comuna,ing,m2,k}));
const OMS=9, R=9, CLP=d=>"$"+Math.round(d).toLocaleString("es-CL");
const active=new Set(CATS.map((_,i)=>i));
const BR="#602C2C";

// Moneda: aro exterior + aro interior + signo $
function drawCoin(g,k,r){
  const c=CATS[k];
  g.append("circle").attr("r",r).attr("fill",c.c).attr("stroke",BR).attr("stroke-width",1);
  g.append("circle").attr("r",r*0.74).attr("fill","none").attr("stroke",c.ink).attr("stroke-opacity",.6).attr("stroke-width",.9);
  g.append("text").text("$").attr("text-anchor","middle").attr("dy","0.35em").attr("fill",c.ink)
   .attr("font-size",r*1.05).attr("font-weight",800).attr("font-family","Figtree,sans-serif").style("pointer-events","none");
}

// Leyenda interactiva
const leg=d3.select("#legend");
CATS.forEach((c,i)=>{
  const b=leg.append("button").attr("class","chip").attr("type","button").attr("aria-pressed","true");
  const s=b.append("svg").attr("width",20).attr("height",20).attr("viewBox","-10 -10 20 20");
  drawCoin(s.append("g"),i,9);
  b.append("span").text(c.n);
  b.on("click",()=>{
    if(active.has(i)){ if(active.size>1) active.delete(i); } else active.add(i);
    leg.selectAll(".chip").each(function(_,j){const on=active.has(j);d3.select(this).classed("off",!on).attr("aria-pressed",on)});
    update();
  });
});

const root=d3.select("#chart"), tip=d3.select("#tip");
let svg;

function render(){
  root.select("svg").remove();
  const W=Math.max(320,root.node().clientWidth), H=Math.round(Math.min(520,Math.max(380,W*0.6)));
  const m={t:18,r:22,b:54,l:56}, iw=W-m.l-m.r, ih=H-m.t-m.b;
  const x=d3.scaleLinear().domain([500000,2250000]).range([0,iw]);
  const y=d3.scaleLinear().domain([0,22]).range([ih,0]);
  svg=root.insert("svg","#tip").attr("class","main").attr("viewBox",`0 0 ${W} ${H}`).attr("role","img")
    .attr("aria-label","Gráfico de dispersión de ingreso promedio versus áreas verdes por habitante");
  const g=svg.append("g").attr("transform",`translate(${m.l},${m.t})`);
  const xt=W<560?[600000,1000000,1400000,1800000,2200000]:d3.range(600000,2300000,200000);
  g.append("g").attr("class","grid").call(d3.axisLeft(y).ticks(5).tickSize(-iw).tickFormat(""));
  g.append("g").attr("class","axis").attr("transform",`translate(0,${ih})`)
    .call(d3.axisBottom(x).tickValues(xt).tickFormat(d=>W<560?"$"+(d/1e6).toFixed(1).replace(".",",")+"M":CLP(d)));
  g.append("g").attr("class","axis").call(d3.axisLeft(y).ticks(5));
  g.append("text").attr("class","atitle").attr("x",iw/2).attr("y",ih+44).attr("text-anchor","middle").text("Ingreso promedio imponible (CLP)");
  g.append("text").attr("class","atitle").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-40).attr("text-anchor","middle").text("Áreas verdes (m² por habitante)");
  // Línea OMS
  g.append("line").attr("x1",0).attr("x2",iw).attr("y1",y(OMS)).attr("y2",y(OMS)).attr("stroke",CATS[4].c).attr("stroke-width",1.5).attr("stroke-dasharray","6 4");
  g.append("text").attr("class","who").attr("x",iw).attr("y",y(OMS)-6).attr("text-anchor","end").text("Recomendación OMS: 9 m²/hab");

  // Posiciones con repulsión para evitar solapamiento
  const nodes=DATA.map(d=>({...d,tx:x(d.ing),ty:y(d.m2),x:x(d.ing),y:y(d.m2)}));
  const sim=d3.forceSimulation(nodes)
    .force("x",d3.forceX(d=>d.tx).strength(.35)).force("y",d3.forceY(d=>d.ty).strength(.35))
    .force("c",d3.forceCollide(R+1.6).strength(1).iterations(4)).stop();
  for(let i=0;i<400;i++){sim.tick();nodes.forEach(n=>{n.x=Math.max(R,Math.min(iw-R,n.x));n.y=Math.max(R,Math.min(ih-R,n.y))})}

  const coins=g.append("g").selectAll("g").data(nodes).join("g").attr("class","coin")
    .attr("transform",d=>`translate(${d.x},${d.y})`).attr("tabindex",0)
    .attr("aria-label",d=>`${d.comuna}: ${CLP(d.ing)}, ${d.m2.toFixed(2)} metros cuadrados por habitante`);
  coins.each(function(d){drawCoin(d3.select(this).append("g").attr("class","s"),d.k,R)});

  const show=(e,d)=>{
    const [px,py]=d3.pointer(e,root.node());
    tip.html(`<b>${d.comuna}</b><span>Ingreso promedio:</span> ${CLP(d.ing)}<br><span>Áreas verdes:</span> ${d.m2.toFixed(2).replace(".",",")} m²/hab<br><span>Prioridad social:</span> ${CATS[d.k].n.replace(" Prioridad Social","")}`);
    const tw=tip.node().offsetWidth;
    tip.style("left",Math.min(W-tw-4,Math.max(4,px+14))+"px").style("top",Math.max(4,py-70)+"px").style("opacity",1);
    coins.style("opacity",c=>c===d?1:(active.has(c.k)?.35:0));
    d3.select(e.currentTarget).raise().select(".s").attr("transform","scale(1.5)");
  };
  const hide=e=>{tip.style("opacity",0);d3.select(e.currentTarget).select(".s").attr("transform",null);update()};
  coins.on("mousemove",(e,d)=>show(e,d)).on("mouseleave",hide)
       .on("focus",(e,d)=>{const r=e.currentTarget.getBoundingClientRect();show({currentTarget:e.currentTarget,clientX:r.left+r.width/2,clientY:r.top+r.height/2},d)}).on("blur",hide);
  update();
}

function update(){
  svg.selectAll(".coin").style("opacity",d=>active.has(d.k)?1:0).style("pointer-events",d=>active.has(d.k)?"all":"none");
}

render();
let rt;addEventListener("resize",()=>{clearTimeout(rt);rt=setTimeout(render,120)});
</script>
</body>
</html>
"""

In [ ]:
# 4) Mostrar el gráfico y guardarlo como HTML
from IPython.display import HTML, display

html = PLANTILLA.replace("__DATOS__", json.dumps(filas, ensure_ascii=False, separators=(",", ":")))

with open("ingresos_areas_verdes.html", "w", encoding="utf-8") as f:
    f.write(html)

display(HTML(html))

In [ ]:
# 5) Descargar el HTML (solo en Colab)
try:
    from google.colab import files
    files.download("ingresos_areas_verdes.html")
except ImportError:
    print("Archivo guardado como ingresos_areas_verdes.html")